In [4]:
from app.rag.embeddings import create_embedding
from app.rag.retrieval import search_chunks

In [6]:
query = "How to install pgvector on Linux?"
query_embedding = create_embedding(query)

print(len(query_embedding))

1024


In [8]:
import os

from sqlmodel import Session
os.environ["DATABASE_URL"] = (
    "postgresql+psycopg://"
    "document_agent:document_agent@127.0.0.1:5432/document_agent"
)
from app.db import engine

with Session(engine) as session:
    results = search_chunks(
        session=session,
        query_embedding=query_embedding,
        limit=5,
    )

    for chunk, distance in results:
        print("=" * 80)
        print("Distance:", distance)
        print("Heading:", chunk.heading_path)
        print(chunk.content[:500])

Distance: 0.16997555075699378
Heading: ['pgvector', 'Languages']
Use pgvector from any language with a Postgres client. You can even generate and store vectors in one language and query them in another.
Distance: 0.1772243561690774
Heading: ['pgvector', 'Additional Installation Methods', 'PGXN']
Install from the [PostgreSQL Extension Network](https://pgxn.org/dist/vector) with:

```sh
pgxn install vector
```
Distance: 0.1958997096174836
Heading: ['pgvector', 'Hosted Postgres']
pgvector is available on [these providers](https://github.com/pgvector/pgvector/issues/54).
Distance: 0.2269424440939115
Heading: ['pgvector']
Open-source vector similarity search for Postgres

Store your vectors with the rest of your data. Supports:

- exact and approximate nearest neighbor search
- single-precision, half-precision, binary, and sparse vectors
- L2 distance, inner product, cosine distance, L1 distance, Hamming distance, and Jaccard distance
- any [language](#languages) with a Postgres client

Plu

In [9]:
query = "Compile and install pgvector on Linux"
query_embedding = create_embedding(query)

with Session(engine) as session:
    results = search_chunks(
        session=session,
        query_embedding=query_embedding,
        limit=5,
    )

    for chunk, distance in results:
        print("=" * 80)
        print("Distance:", distance)
        print("Heading:", chunk.heading_path)
        print(chunk.content[:300])

Distance: 0.17285664324253092
Heading: ['pgvector', 'Additional Installation Methods', 'PGXN']
Install from the [PostgreSQL Extension Network](https://pgxn.org/dist/vector) with:

```sh
pgxn install vector
```
Distance: 0.18983724450434059
Heading: ['pgvector', 'Languages']
Use pgvector from any language with a Postgres client. You can even generate and store vectors in one language and query them in another.
Distance: 0.20208545917017562
Heading: ['pgvector', 'Hosted Postgres']
pgvector is available on [these providers](https://github.com/pgvector/pgvector/issues/54).
Distance: 0.2327438413042684
Heading: ['pgvector', 'Installation', 'Linux and Mac']
Compile and install the extension (supports Postgres 13+)

```sh
cd /tmp
git clone --branch v0.8.6 https://github.com/pgvector/pgvector.git
cd pgvector
make
make install # may need sudo
```

See the [installation notes](#installation-notes---linux-and-mac) if you run into issues

You can also instal
Distance: 0.2352780752298318
Heading: [

In [10]:
chunk = results[0][0]
embedding_text = (
    " > ".join(chunk.heading_path)
    + "\n\n"
    + chunk.content
)

print(embedding_text)

pgvector > Additional Installation Methods > PGXN

Install from the [PostgreSQL Extension Network](https://pgxn.org/dist/vector) with:

```sh
pgxn install vector
```


In [7]:
query = "Compile and install pgvector on Linux"

query_embedding = create_embedding(query)

In [8]:
chunk_embedding = create_embedding(embedding_text)

In [9]:
from sqlmodel import Session, select

from app.db import engine
from app.models import DocumentChunk

with Session(engine) as session:
    chunk = session.exec(
        select(DocumentChunk)
        .where(DocumentChunk.id == results[0][0].id)
    ).one()

    old_embedding = chunk.embedding

In [10]:
from app.rag.embeddings import create_embedding

embedding_text = (
    " > ".join(chunk.heading_path)
    + "\n\n"
    + chunk.content
)

new_embedding = create_embedding(embedding_text)

print(len(old_embedding))
print(len(new_embedding))

1024
1024


In [15]:
import numpy as np

query = np.array(query_embedding)
old = np.array(old_embedding)
new = np.array(new_embedding)

old_similarity = np.dot(query, old) / (
    np.linalg.norm(query) * np.linalg.norm(old)
)

new_similarity = np.dot(query, new) / (
    np.linalg.norm(query) * np.linalg.norm(new)
)

old_distance = 1 - old_similarity
new_distance = 1 - new_similarity

print("Без заголовка:", old_distance)
print("С заголовком:", new_distance)

Без заголовка: 0.23566570616968197
С заголовком: 0.21859457633035395


In [13]:
with Session(engine) as session:
    linux_chunk = next(
    chunk
    for chunk, distance in results
    if "Linux and Mac" in chunk.heading_path
)

print(linux_chunk.heading_path)
print(linux_chunk.content[:200])

['pgvector', 'Installation', 'Linux and Mac']
Compile and install the extension (supports Postgres 13+)

```sh
cd /tmp
git clone --branch v0.8.6 https://github.com/pgvector/pgvector.git
cd pgvector
make
make install # may need sudo
```

See the [


In [14]:
old_embedding = linux_chunk.embedding

embedding_text = (
    " > ".join(linux_chunk.heading_path)
    + "\n\n"
    + linux_chunk.content
)

new_embedding = create_embedding(embedding_text)

In [11]:
with Session(engine) as session:
    results = search_chunks(
        session=session,
        query_embedding=query_embedding,
        limit=20,
    )

    for rank, (chunk, distance) in enumerate(results, start=1):
        print(
            f"{rank:2}. "
            f"distance={distance:.4f} | "
            f"heading={chunk.heading_path}"
        )

 1. distance=0.1729 | heading=['pgvector', 'Additional Installation Methods', 'PGXN']
 2. distance=0.1898 | heading=['pgvector', 'Languages']
 3. distance=0.2021 | heading=['pgvector', 'Hosted Postgres']
 4. distance=0.2327 | heading=['pgvector', 'Installation', 'Linux and Mac']
 5. distance=0.2353 | heading=['pgvector', 'Additional Installation Methods', 'APK']
 6. distance=0.2396 | heading=['pgvector', 'Additional Installation Methods', 'Yum']
 7. distance=0.2465 | heading=['pgvector', 'Installation', 'Windows']
 8. distance=0.2504 | heading=['pgvector', 'Additional Installation Methods', 'Homebrew']
 9. distance=0.2544 | heading=['pgvector', 'Languages']
10. distance=0.2547 | heading=['pgvector', 'Languages']
11. distance=0.2564 | heading=['pgvector']
12. distance=0.2609 | heading=['pgvector', 'Additional Installation Methods', 'conda-forge']
13. distance=0.2655 | heading=['pgvector', 'Contributing']
14. distance=0.2760 | heading=['pgvector', 'Additional Installation Methods', 'pkg'

In [12]:
queries = [
    "How to install pgvector on Linux?",
    "Install pgvector Linux",
    "Compile and install the extension on Linux",
]

In [13]:
for query in queries:
    query_embedding = create_embedding(query)

    with Session(engine) as session:
        results = search_chunks(
            session=session,
            query_embedding=query_embedding,
            limit=5,
        )

    print("\nQUERY:", query)

    for chunk, distance in results:
        print(
            f"{distance:.4f} | {chunk.heading_path}"
        )


QUERY: How to install pgvector on Linux?
0.1708 | ['pgvector', 'Languages']
0.1780 | ['pgvector', 'Additional Installation Methods', 'PGXN']
0.1964 | ['pgvector', 'Hosted Postgres']
0.2278 | ['pgvector']
0.2422 | ['pgvector', 'Additional Installation Methods', 'Yum']

QUERY: Install pgvector Linux
0.1920 | ['pgvector', 'Additional Installation Methods', 'PGXN']
0.2082 | ['pgvector', 'Hosted Postgres']
0.2090 | ['pgvector', 'Languages']
0.2429 | ['pgvector', 'Additional Installation Methods', 'APK']
0.2445 | ['pgvector', 'Additional Installation Methods', 'Yum']

QUERY: Compile and install the extension on Linux
0.3639 | ['pgvector', 'Installation', 'Linux and Mac']
0.4259 | ['pgvector', 'Additional Installation Methods', 'PGXN']
0.4476 | ['pgvector', 'Installation Notes - Linux and Mac', 'Portability']
0.5178 | ['pgvector', 'Upgrading']
0.5240 | ['pgvector', 'Installation Notes - Windows', 'Missing Header']


In [15]:
query = "How do I compile and install pgvector from source on Linux?"

query_with_instruction = (
    "Instruct: Given a document query, retrieve the most relevant chunk.\n"
    f"Query: {query}"
)

query_embedding = create_embedding(query_with_instruction)

with Session(engine) as session:
    results = search_chunks(
        session=session,
        query_embedding=query_embedding,
        limit=10,
    )

    for rank, (chunk, distance) in enumerate(results, start=1):
        print(
            f"{rank:2}. "
            f"distance={distance:.4f} | "
            f"heading={chunk.heading_path}"
        )

 1. distance=0.2240 | heading=['pgvector', 'Installation', 'Windows']
 2. distance=0.2280 | heading=['pgvector', 'Installation', 'Linux and Mac']
 3. distance=0.2289 | heading=['pgvector', 'Additional Installation Methods', 'PGXN']
 4. distance=0.2546 | heading=['pgvector', 'Additional Installation Methods', 'Yum']
 5. distance=0.2604 | heading=['pgvector', 'Additional Installation Methods', 'conda-forge']
 6. distance=0.2658 | heading=['pgvector', 'Languages']
 7. distance=0.2663 | heading=['pgvector', 'Languages']
 8. distance=0.2760 | heading=['pgvector', 'Installation Notes - Linux and Mac', 'Portability']
 9. distance=0.2769 | heading=['pgvector', 'Hosted Postgres']
10. distance=0.2782 | heading=['pgvector', 'Languages']


In [16]:
from sqlalchemy import text

query = "How do I compile and install pgvector from source on Linux?"

with Session(engine) as session:
    result = session.exec(
        text("""
            SELECT
                id,
                chunk_index,
                heading_path,
                ts_rank(
                    to_tsvector('english', content),
                    plainto_tsquery('english', :query)
                ) AS rank
            FROM document_chunks
            WHERE embedding IS NOT NULL
            ORDER BY rank DESC
            LIMIT 10
        """),
        params={"query": query},
    ).all()

    for row in result:
        print(
            f"rank={row.rank:.4f} | "
            f"heading={row.heading_path}"
        )

rank=0.4176 | heading=['pgvector', 'Installation', 'Linux and Mac']
rank=0.3188 | heading=['pgvector', 'Additional Installation Methods', 'Yum']
rank=0.1781 | heading=['pgvector', 'Additional Installation Methods', 'pkg']
rank=0.1272 | heading=['pgvector', 'Installation', 'Windows']
rank=0.0995 | heading=['pgvector', 'Installation Notes - Linux and Mac', 'Portability']
rank=0.0991 | heading=['pgvector', 'Additional Installation Methods', 'Homebrew']
rank=0.0974 | heading=['pgvector', 'Contributing']
rank=0.0974 | heading=['pgvector', 'Additional Installation Methods', 'APT']
rank=0.0968 | heading=['pgvector', 'Additional Installation Methods', 'conda-forge']
rank=0.0398 | heading=['pgvector', 'Additional Installation Methods', 'APK']


In [2]:
import os
from sqlmodel import Session
os.environ["DATABASE_URL"] = (
    "postgresql+psycopg://"
    "document_agent:document_agent@127.0.0.1:5432/document_agent"
)
from app.db import engine
from app.rag.retrieval import hybrid_search

query = "How do I compile and install pgvector from source on Linux?"

with Session(engine) as session:
    results = hybrid_search(
        session=session,
        query=query,
        limit=10,
    )

for rank, (chunk, score, vector_rank, text_rank) in enumerate(results, start=1):
    print(
        f"{rank:2}. "
        f"score={score:.6f} | "
        f"vector={vector_rank} | "
        f"text={text_rank} | "
        f"{chunk.heading_path}"
    )

 1. score=0.031281 | vector=6 | text=2 | ['pgvector', 'Additional Installation Methods', 'Yum']
 2. score=0.030550 | vector=7 | text=4 | ['pgvector', 'Installation', 'Windows']
 3. score=0.030478 | vector=11 | text=1 | ['pgvector', 'Installation', 'Linux and Mac']
 4. score=0.029643 | vector=2 | text=14 | ['pgvector', 'Additional Installation Methods', 'PGXN']
 5. score=0.029418 | vector=9 | text=7 | ['pgvector', 'Contributing']
 6. score=0.029031 | vector=16 | text=3 | ['pgvector', 'Additional Installation Methods', 'pkg']
 7. score=0.028898 | vector=14 | text=5 | ['pgvector', 'Installation Notes - Linux and Mac', 'Portability']
 8. score=0.028850 | vector=13 | text=6 | ['pgvector', 'Additional Installation Methods', 'Homebrew']
 9. score=0.028571 | vector=10 | text=10 | ['pgvector', 'Additional Installation Methods', 'APK']
10. score=0.028382 | vector=12 | text=9 | ['pgvector', 'Additional Installation Methods', 'conda-forge']


In [ ]:
from app.rag.retrieval import search_chunks, search_text_chunks

query = "How do I compile and install pgvector from source on Linux?"

with Session(engine) as session:
    vector_results = search_chunks(
        session=session,
        query_embedding=create_embedding(query),
        limit=20,
    )

    text_results = search_text_chunks(
        session=session,
        query=query,
        limit=20,
    )

NameError: name 'search_chunks' is not defined